# 02 - MCMC Basics for Bayesian Inference

In the previous notebook, we saw that exact posterior inference is often intractable.
This notebook introduces **MCMC** as a sampling-based solution.

We will learn:
1. Why MCMC is useful
2. How Metropolis-Hastings works
3. How to diagnose whether samples are useful
4. Common pitfalls and tuning intuition

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## Part 1: Why MCMC?

Recall Bayes' theorem:
$$p(\theta \mid x) = \frac{p(x \mid \theta)p(\theta)}{p(x)}$$

The challenge is the evidence term:
$$p(x) = \int p(x \mid \theta)p(\theta) \, d\theta$$

When this integral is hard, MCMC gives us samples from the posterior (approximately),
without computing the normalizing constant directly.

## Part 2: Metropolis-Hastings in plain English

At each step:
1. Propose a new point $\theta'$ near current $\theta$
2. Compute acceptance ratio
$$a = \min\left(1, \frac{\tilde{p}(\theta')}{\tilde{p}(\theta)}\right)$$
   (for symmetric proposals)
3. Accept with probability $a$, otherwise stay put

Here $\tilde{p}$ is an **unnormalized** target density, e.g., prior $\times$ likelihood.

In [ ]:
# We'll use a 1D bimodal target distribution so behavior is easy to visualize
def target_unnormalized(theta):
    # Mixture of two Gaussians (unnormalized scale is fine for MH)
    peak1 = 0.35 * stats.norm.pdf(theta, loc=-2.0, scale=0.7)
    peak2 = 0.65 * stats.norm.pdf(theta, loc=2.5, scale=1.0)
    return peak1 + peak2

grid = np.linspace(-6, 7, 1000)
density = target_unnormalized(grid)

plt.figure(figsize=(9, 4))
plt.plot(grid, density, color='navy', linewidth=2)
plt.title('Target distribution (unnormalized)')
plt.xlabel('theta')
plt.ylabel('density (up to constant)')
plt.show()

In [ ]:
def metropolis_hastings(n_samples, proposal_std, initial_theta=0.0):
    samples = np.zeros(n_samples)
    theta = initial_theta
    accepts = 0

    for i in range(n_samples):
        # Random-walk proposal: theta' = theta + epsilon
        proposal = theta + np.random.normal(0, proposal_std)

        # Acceptance probability (symmetric proposal => proposal terms cancel)
        current_val = target_unnormalized(theta)
        proposal_val = target_unnormalized(proposal)
        acceptance_ratio = proposal_val / current_val
        acceptance_prob = min(1.0, acceptance_ratio)

        if np.random.rand() < acceptance_prob:
            theta = proposal
            accepts += 1

        samples[i] = theta

    acceptance_rate = accepts / n_samples
    return samples, acceptance_rate

In [ ]:
n_samples = 12000
proposal_std = 1.0
samples, acc_rate = metropolis_hastings(n_samples=n_samples, proposal_std=proposal_std, initial_theta=-5.0)

burn_in = 2000
post_burn_samples = samples[burn_in:]

print(f'Acceptance rate: {acc_rate:.3f}')
print(f'Using {len(post_burn_samples)} samples after burn-in')

## Part 3: Diagnosing the chain

Three core diagnostics:
- **Trace plot**: Does the chain explore the space?
- **Histogram vs target**: Does sample distribution match target shape?
- **Autocorrelation**: Are successive samples too similar?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Trace plot
axes[0].plot(samples[:3000], color='teal', linewidth=1)
axes[0].axvline(burn_in, color='red', linestyle='--', label='burn-in cutoff')
axes[0].set_title('Trace plot (first 3000 iterations)')
axes[0].set_xlabel('iteration')
axes[0].set_ylabel('theta')
axes[0].legend()

# Histogram + target
axes[1].hist(post_burn_samples, bins=60, density=True, alpha=0.6, color='coral', edgecolor='black', label='MCMC samples')
axes[1].plot(grid, density, color='navy', linewidth=2, label='target (unnormalized)')
axes[1].set_title('Posterior approximation via samples')
axes[1].set_xlabel('theta')
axes[1].set_ylabel('density')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
def autocorrelation(x, max_lag=60):
    x = np.asarray(x)
    x = x - np.mean(x)
    var = np.var(x)
    acf = [1.0]
    for lag in range(1, max_lag + 1):
        cov = np.mean(x[:-lag] * x[lag:])
        acf.append(cov / var)
    return np.array(acf)

acf_vals = autocorrelation(post_burn_samples, max_lag=60)

plt.figure(figsize=(9, 4))
plt.stem(range(len(acf_vals)), acf_vals, basefmt=' ', linefmt='tab:blue', markerfmt='o')
plt.title('Autocorrelation of post burn-in chain')
plt.xlabel('lag')
plt.ylabel('autocorrelation')
plt.ylim(-0.1, 1.05)
plt.show()

## Part 4: Proposal scale tuning intuition

- Too small proposal std: high acceptance, but tiny moves (slow exploration)
- Too large proposal std: low acceptance, many rejections (also slow)
- Somewhere in the middle usually works best

In [ ]:
proposal_values = [0.2, 1.0, 3.0]
results = {}

for ps in proposal_values:
    s, a = metropolis_hastings(n_samples=8000, proposal_std=ps, initial_theta=-5.0)
    results[ps] = (s[1000:], a)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, ps in zip(axes, proposal_values):
    s, a = results[ps]
    ax.hist(s, bins=50, density=True, alpha=0.65, color='skyblue', edgecolor='black')
    ax.plot(grid, density, color='navy', linewidth=2)
    ax.set_title(f'proposal_std={ps}\naccept={a:.2f}')
    ax.set_xlabel('theta')

axes[0].set_ylabel('density')
plt.suptitle('Effect of proposal scale on sampling quality', y=1.03)
plt.tight_layout()
plt.show()

## Summary

What to remember:
1. MCMC turns integration into sampling
2. Metropolis-Hastings only needs unnormalized density
3. Burn-in and autocorrelation matter for sample quality
4. Proposal tuning is crucial for efficient exploration

In the next notebook, we can cover one of these paths:
- Hamiltonian Monte Carlo (HMC) intuition
- Gibbs sampling
- Variational Inference start: KL divergence and ELBO

In [ ]:
# Optional exercises
# 1) Try starting point initial_theta=5.0 and compare trace plots.
# 2) Increase n_samples to 50000. How does the histogram improve?
# 3) Change the target to a single Gaussian and inspect autocorrelation.

pass